**This Notebook contain raw data loading <> Initial inspection of Data and Data inconsistencies fix -changing inproper column name if any , data type change of Dataframe after parsing dataframes from csv files so that if we load data into database in SQL server the column type is converted as expected column type in SQL and finally this notebook is saving processed dataframes into Parquet to be load and used in further pipeline**  

In [ ]:
import pandas as pd
import numpy as np
import sklearn.preprocessing
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACT_DIR = PROJECT_ROOT / "artifact"
ARTIFACT_DIR.mkdir(exist_ok=True)

customers = pd.read_csv(DATA_DIR / 'customers.csv', encoding='utf-8')
orders = pd.read_csv(
    DATA_DIR / 'orders.csv',
    encoding='utf-8',
    parse_dates=[
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date',
    ],
)
order_items = pd.read_csv(DATA_DIR / 'order_items.csv', encoding='utf-8', parse_dates=['shipping_limit_date'])
sellers = pd.read_csv(DATA_DIR / 'sellers.csv', encoding='utf-8')
products = pd.read_csv(DATA_DIR / 'products.csv', encoding='utf-8')
payments = pd.read_csv(DATA_DIR / 'order_payments.csv', encoding='utf-8')
reviews = pd.read_csv(
    DATA_DIR / 'order_reviews.csv',
    encoding='utf-8',
    parse_dates=['review_creation_date', 'review_answer_timestamp'],
)
geolocations = pd.read_csv(DATA_DIR / 'geolocation.csv', encoding='utf-8')


In [ ]:

 # In products some columns has incorrect spelling name 
print(products.columns.to_list())
# Product_name_lenght -Incorrect (basically length incorrect at some places )

products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
}, inplace=True)
print("After fixing: ",products.columns.to_list())


In [ ]:

##3 DTYPE CORRECTION - OLIST

## i have Just changed otherwise this step if we can skip  
# ---------- CUSTOMERS ----------

customers = customers.astype({
    "customer_id": "string",
    "customer_unique_id": "string",
    "customer_city": "string",
    "customer_state": "string",
    "customer_zip_code_prefix": "string"
})


# ---------- SELLERS ----------

sellers = sellers.astype({
    "seller_id": "string",
    "seller_city": "string",
    "seller_state": "string",
    "seller_zip_code_prefix": "string"
})


# ---------- PRODUCTS ----------

products = products.astype({
    "product_id": "string",
    "product_category_name": "string",

    "product_name_length": "Int32",
    "product_description_length": "Int32",
    "product_photos_qty": "Int16",

    "product_weight_g": "float32",
    "product_length_cm": "float32",
    "product_height_cm": "float32",
    "product_width_cm": "float32"
})


# ---------- ORDERS ----------

orders = orders.astype({
    "order_id": "string",
    "customer_id": "string",
    "order_status": "string"
})


# ---------- ORDER ITEMS ----------

order_items = order_items.astype({
    "order_id": "string",
    "product_id": "string",
    "seller_id": "string",

    "order_item_id": "Int16",

    "price": "float32",
    "freight_value": "float32"
})




# ---------- PAYMENTS ----------

payments = payments.astype({
    "order_id": "string",
    "payment_type": "string",

    "payment_sequential": "Int16",
    "payment_installments": "Int16",

    "payment_value": "float32"
})


# ---------- REVIEWS ----------

reviews = reviews.astype({
    "review_id": "string",
    "order_id": "string",

    "review_comment_title": "string",
    "review_comment_message": "string",

    "review_score": "Int8"
})




# ---------- GEOLOCATIONS ----------

geolocations = geolocations.astype({
    "geolocation_zip_code_prefix": "string",
    "geolocation_city": "string",
    "geolocation_state": "string",

    "geolocation_lat": "float32",
    "geolocation_lng": "float32"
})


# ==============================
# UPDATE DFS DICTIONARY
# ==============================

dfs = {
    "customers": customers,
    "sellers": sellers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "geolocations": geolocations
}


# ==============================
# CHECK FINAL DTYPES
# ==============================

for name, df in dfs.items():
    print(f"\n{name.upper()}")
    print(df.dtypes)
    

In [ ]:
id=[]
for name ,df in dfs.items():
    print(f"  {name}  processing     ")
    for col in df:
        if df[col].nunique()==df.drop_duplicates().shape[0]:
            id.append(col)
   
print(id)

In [ ]:
import os

# Folder path
save_path = ARTIFACT_DIR

# Create folder if not exists
ARTIFACT_DIR.mkdir(exist_ok=True)

# Save all cleaned dfs

for name, df in dfs.items():

    file_path = ARTIFACT_DIR / f"{name}_clean.parquet"
    
    for col in df.select_dtypes(include=['Int8','Int16','Int32','Int64']).columns:
        df[col] = df[col].astype('float32')   # nullable int → float keeps NaN
    df.to_parquet(file_path, index=False)
    print(f"Saved  {name:<15} → {file_path}")
    df.to_parquet(
        file_path,
        engine="pyarrow",
        compression="snappy",
        index=False
    )

    print(f"{name} saved successfully")

## Why Parquet ?


In [ ]:
dfs_clean={}
for name,df in dfs.items():
    new_name=f"{name}_clean"
    dfs_clean[new_name]=df.copy()
    
# for name,df in dfs_clean.items():
#     print(f"---processing {name}-----")
#     for col in df.columns.to_list():
#         print(col)

for name ,df in dfs_clean.items():
    globals()[name]=df
    
reviews_clean.columns
